# Text classification with Generative Models

Objectives : compare several approaches of feeling classifiers (specialized model, embeddings + classifier, no label methods, zero-shot, generative models)
We will evaluate each model with classic metrics: precision, recall, F1 score, accuracy



In [1]:

!pip -q install "transformers>=4.41.2" "accelerate>=0.31.0" datasets sentence-transformers scikit-learn tqdm

You should consider upgrading via the '/Users/othmanhamoumi/Desktop/Oulaya NLP/env/bin/python3 -m pip install --upgrade pip' command.


We install necessary libraries: Transformers, Datasets, Sentence-Transformers, sklearn

In [2]:
import numpy as np
from tqdm import tqdm

from datasets import load_dataset
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset

from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity

/Users/othmanhamoumi/Desktop/Oulaya NLP/env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/othmanhamoumi/Desktop/Oulaya NLP/env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We load the Rotten Tomatoes dataset: binary feeling: 0 = negative, 1 = positive and we look for the splits

In [3]:
data = load_dataset("rotten_tomatoes")
data

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

We define an standard evaluation function with classification_report (precision/recall/F1 + accuracy).


In [4]:
def evaluate_performance(y_true, y_pred):
    report = classification_report(
        y_true, y_pred,
        target_names=["Negative Review", "Positive Review"]
    )
    print(report)

We test a model which is already trained for the feeling and we evaluate it in the test set

In [5]:
import sys
!{sys.executable} -m pip install -U pip
!{sys.executable} -m pip install torch

zsh:1: no such file or directory: /Users/othmanhamoumi/Desktop/Oulaya
zsh:1: no such file or directory: /Users/othmanhamoumi/Desktop/Oulaya


In [6]:
import sys
!{sys.executable} -m pip install -U pip
!{sys.executable} -m pip install -U transformers accelerate sentencepiece

zsh:1: no such file or directory: /Users/othmanhamoumi/Desktop/Oulaya
zsh:1: no such file or directory: /Users/othmanhamoumi/Desktop/Oulaya


In [7]:
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

pipe = pipeline(
    task="text-classification",
    model=model_path,
    tokenizer=model_path,
    top_k=None,
    device=device
)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


In [8]:
from datasets import load_dataset

data = load_dataset("rotten_tomatoes")


In [9]:
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset


y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "text")), total=len(data["test"])):
    neg = output[0]["score"]
    pos = output[2]["score"]
    y_pred.append(int(np.argmax([neg, pos])))

evaluate_performance(data["test"]["label"], y_pred)

100%|██████████| 1066/1066 [00:34<00:00, 31.26it/s]

                 precision    recall  f1-score   support

Negative Review       0.50      1.00      0.67       533
Positive Review       0.00      0.00      0.00       533

       accuracy                           0.50      1066
      macro avg       0.25      0.50      0.33      1066
   weighted avg       0.25      0.50      0.33      1066




/Users/othmanhamoumi/Desktop/Oulaya NLP/env/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/othmanhamoumi/Desktop/Oulaya NLP/env/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/othmanhamoumi/Desktop/Oulaya NLP/env/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modi

We freeze a model of embeddings (Sentence-Transformers), then we train a small classifier (logistic regression) on these features.

In [10]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

train_embeddings = embedder.encode(data["train"]["text"], show_progress_bar=True)
test_embeddings  = embedder.encode(data["test"]["text"], show_progress_bar=True)

train_embeddings.shape, test_embeddings.shape

Batches: 100%|██████████| 34/34 [00:02<00:00, 16.82it/s]


((8530, 768), (1066, 768))

In [11]:
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(train_embeddings, data["train"]["label"])

y_pred = clf.predict(test_embeddings)
evaluate_performance(data["test"]["label"], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.85      0.86      0.85       533
Positive Review       0.86      0.85      0.85       533

       accuracy                           0.85      1066
      macro avg       0.85      0.85      0.85      1066
   weighted avg       0.85      0.85      0.85      1066



Without training the classifier, we calcul a embedding "prototype" per class (so the mean) and we predict visa cosine similarity

In [12]:
y_train = np.array(data["train"]["label"])
proto_neg = train_embeddings[y_train == 0].mean(axis=0)
proto_pos = train_embeddings[y_train == 1].mean(axis=0)

prototypes = np.vstack([proto_neg, proto_pos])
sim_matrix = cosine_similarity(test_embeddings, prototypes)
y_pred = np.argmax(sim_matrix, axis=1)

evaluate_performance(data["test"]["label"], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.85      0.84      0.84       533
Positive Review       0.84      0.85      0.84       533

       accuracy                           0.84      1066
      macro avg       0.84      0.84      0.84      1066
   weighted avg       0.84      0.84      0.84      1066



Zéro-shot : on décrit les labels en texte ("A negative review", "A positive review"), on embed ces descriptions puis on classe par similarité.
Zero-shot: we describe labels in test ("A negative review", "A positive review"), and we embed this descriptions, then we class them by similarity

In [13]:
label_embeddings = embedder.encode(["A negative review", "A positive review"])

sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

evaluate_performance(data["test"]["label"], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.78      0.77      0.78       533
Positive Review       0.77      0.79      0.78       533

       accuracy                           0.78      1066
      macro avg       0.78      0.78      0.78      1066
   weighted avg       0.78      0.78      0.78      1066



Generatives LLM can classify via prompting (they generate a text output). Here, we are doing a prompt which will return a [0,1] score.

In [14]:
!pip -q install groq python-dotenv #we need it for this

You should consider upgrading via the '/Users/othmanhamoumi/Desktop/Oulaya NLP/env/bin/python3 -m pip install --upgrade pip' command.


In [15]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
print(os.getenv("GROQ_API_KEY"))


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


In [16]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [17]:
sample_text = data["test"]["text"][0]
print(sample_text)

lovingly photographed in the manner of a golden book sprung to life , stuart little 2 manages sweetness largely without stickiness .


In [23]:
def groq_score(review, model="llama-3.3-70b-versatile"):
    messages = [
        {"role": "system", "content": "You are a sentiment classifier. Rate sentiment between 0 (negative) and 1 (positive). Respond with only the number."},
        {"role": "user", "content": f"Rate the sentiment of this movie review: {review}"}
    ]
    r = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
        max_tokens=10
    )
    return r.choices[0].message.content.strip()



In [21]:
import os, requests
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.environ["GROQ_API_KEY"]

r = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {api_key}"}
)
r.raise_for_status()

models = [m["id"] for m in r.json()["data"]]
models[:30], len(models)


(['groq/compound',
  'groq/compound-mini',
  'llama-3.1-8b-instant',
  'moonshotai/kimi-k2-instruct',
  'playai-tts',
  'llama-3.3-70b-versatile',
  'meta-llama/llama-guard-4-12b',
  'openai/gpt-oss-safeguard-20b',
  'qwen/qwen3-32b',
  'allam-2-7b',
  'whisper-large-v3-turbo',
  'openai/gpt-oss-120b',
  'whisper-large-v3',
  'meta-llama/llama-prompt-guard-2-22m',
  'playai-tts-arabic',
  'meta-llama/llama-prompt-guard-2-86m',
  'meta-llama/llama-4-scout-17b-16e-instruct',
  'moonshotai/kimi-k2-instruct-0905',
  'meta-llama/llama-4-maverick-17b-128e-instruct',
  'openai/gpt-oss-20b'],
 20)

In [24]:
score = float(groq_score(sample_text))
pred = 1 if score >= 0.5 else 0
score, pred

(0.9, 1)

If we want to evaluate across the entire test set, we loop through all the texts.

In [25]:
subset_n = 50
pred_scores = [float(groq_score(txt)) for txt in tqdm(data["test"]["text"][:subset_n])]
y_pred = [1 if s >= 0.5 else 0 for s in pred_scores]

evaluate_performance(data["test"]["label"][:subset_n], y_pred)

100%|██████████| 50/50 [00:45<00:00,  1.10it/s]

                 precision    recall  f1-score   support

Negative Review       0.00      0.00      0.00         0
Positive Review       1.00      0.90      0.95        50

       accuracy                           0.90        50
      macro avg       0.50      0.45      0.47        50
   weighted avg       1.00      0.90      0.95        50




/Users/othmanhamoumi/Desktop/Oulaya NLP/env/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/othmanhamoumi/Desktop/Oulaya NLP/env/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/othmanhamoumi/Desktop/Oulaya NLP/env/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitali

With a text-to-text model (Flan-T5), we rephrase the thing we were doing: 'positif or negatif?', then we map the reponse in 0/1

In [26]:
device = 0 if torch.cuda.is_available() else -1

t5_pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    device=device
)

Device set to use cpu


In [27]:
prompt = "Is the following sentence positive or negative? "
data = data.map(lambda ex: {"t5": prompt + ex["text"]})
data["test"][0]["t5"][:200]

Map: 100%|██████████| 1066/1066 [00:00<00:00, 68451.70 examples/s]


'Is the following sentence positive or negative? lovingly photographed in the manner of a golden book sprung to life , stuart little 2 manages sweetness largely without stickiness .'

In [28]:
y_pred = []
for out in tqdm(t5_pipe(KeyDataset(data["test"], "t5")), total=len(data["test"])):
    ans = out[0]["generated_text"].strip().lower()
    y_pred.append(0 if ans == "negative" else 1)

evaluate_performance(data["test"]["label"], y_pred)

100%|██████████| 1066/1066 [00:55<00:00, 19.26it/s]

                 precision    recall  f1-score   support

Negative Review       0.83      0.85      0.84       533
Positive Review       0.85      0.83      0.84       533

       accuracy                           0.84      1066
      macro avg       0.84      0.84      0.84      1066
   weighted avg       0.84      0.84      0.84      1066



## Conclusion

 Classification heads, which are specialized models, produce high-quality results fast.

 Embeddings combined with a small supervised classifier are frequently very good (easy to use and effective).

 Averages, or class prototypes, are already quite effective.

 Although unexpected, zero-shot classification using embeddings is less precise.

 Although they can classify through prompting, generative LLMs are limited by cost, latency, and variability.

 In the absence of an external API, text-to-text models (T5) are a good "generative" compromise.